# 10 — Chat multi-turno: memoria de corto plazo

**Level 1 — LLM Engineering**

El LLM no recuerda nada entre requests: cada llamada es nueva. La "memoria" es el **historial** que reenviamos completo en cada turno.

El experimento: le decimos nuestro nombre, preguntamos otra cosa, y después preguntamos por el nombre — si lo recuerda, el historial funcionó.

In [1]:
import requests

OLLAMA_HOST = "http://localhost:11434"


def chat(messages: list[dict], model: str = "llama3.2") -> str:
    """Envia la conversacion completa a Ollama y devuelve la respuesta."""
    url = f"{OLLAMA_HOST}/api/chat"
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
    }
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]["content"]

## El estado: historial que crece

In [2]:
historial: list[dict] = [
    {
        "role": "system",
        "content": (
            "Eres un asistente amigable en espanol. "
            "Recuerdas la conversacion y respondes con naturalidad."
        ),
    },
]

def turno(texto_usuario: str) -> None:
    """Un turno de conversacion: agrega el mensaje, responde, agrega la respuesta."""
    historial.append({"role": "user", "content": texto_usuario})
    respuesta = chat(historial)
    historial.append({"role": "assistant", "content": respuesta})
    print(f"Tu> {texto_usuario}")
    print(f"Bot> {respuesta}")
    print(f"[historial: {len(historial)} mensajes]\n")

## Turno 1 — Presentación

In [3]:
turno("Hola, me llamo Carlos y trabajo en datos.")

Tu> Hola, me llamo Carlos y trabajo en datos.
Bot> ¡Hola Carlos! Me alegra conocerte. Eres un experto en datos, ¿verdad? ¿Te gusta tu trabajo? Estoy aquí para ayudarte con cualquier cosa que necesites. ¿Qué te trae hoy?
[historial: 3 mensajes]



## Turno 2 — Pregunta sin relación

In [4]:
turno("Cual es la capital de Francia?")

Tu> Cual es la capital de Francia?
Bot> Una pregunta fácil, pero muy común! La respuesta es París. ¿Necesitas saber algo más sobre Francia o es solo un dato que necesitas recordar?
[historial: 5 mensajes]



## Turno 3 — La prueba de memoria

In [5]:
turno("Como me llamo?")

Tu> Como me llamo?
Bot> Me acuerdo! Me dijiste que te llamas Carlos. ¡No te preocupes, es fácil olvidar en una conversación larga! ¿Quieres seguir hablando sobre algo en particular o simplemente charlar un rato?
[historial: 7 mensajes]



## Conclusión

Si en el turno 3 recuerda "Carlos", la memoria de corto plazo (reenvío del historial) funcionó. Sin ese reenvío, el modelo no tendría forma de saberlo.